<a href="https://colab.research.google.com/github/LiuChen-5749342/Generative-AI-and-AI-Applications/blob/main/Task_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Required Task 10
Your task is to utilize the Gemini VLM to predict the `total_price` for a subset of receipts and then evaluate the model's performance against the ground truth `total_price` already present in `df_receipt`.

#### Instructions:

1.  **Randomly Select 15 Records**:
    *   From the `df_receipt` DataFrame, randomly select 100 receipts. This will be your test set for Gemini's prediction.
    *   Store these 15 records in a new DataFrame, say `df_test_receipts`.

2.  **Define a Prompt for Gemini**:
    *   Create a clear and concise prompt that instructs Gemini to extract only the `total_price` from a given receipt image. Emphasize that the output should be a single numerical value (float).
    *   Example prompt: `"Extract the total amount from this receipt. Provide only the numerical value as a float."`

3.  **Process `df_test_receipts` with Gemini**:
    *   Iterate through each row in `df_test_receipts`.
    *   For each receipt's image, call the Gemini VLM with your defined prompt.
    *   Parse Gemini's response to extract the predicted `total_price`. Handle potential errors (e.g., non-numeric responses, API issues) by assigning `None` or `NaN` if a valid price cannot be extracted.
    *   Add the extracted prediction as a new column, `predicted_total_price`, to `df_test_receipts`.

4.  **Evaluate Predictions**:
    *   Compare the `predicted_total_price` with the `total_price` (ground truth) in `df_test_receipts`.
    *   Calculate appropriate evaluation metrics. Consider the following:
        *   **Mean Absolute Error (MAE)**: Average of the absolute differences between predicted and actual values.
        *   **Number of successful extractions**: Count how many predictions were successfully extracted (not `None` or `NaN`).
        *   **Accuracy within a threshold**: Calculate the percentage of predictions that are within a certain percentage (e.g., 5% or 10%) of the ground truth.

5.  **Display Results**:
    *   Print the calculated evaluation metrics.
    *   Display `df_test_receipts` with `total_price` and `predicted_total_price` columns for a few sample rows to show the comparison.


This code cell loads the API key from Colab secrets and sets it as an environment variable for the Gemini API.

In [1]:
from google.colab import userdata
import os

# Load the API key from Colab secrets, assuming it's stored as 'GEMINI_API_KEY'
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

# Set the environment variable for the Gemini API
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API key loaded and environment variable set.")

API key loaded and environment variable set.


This code block imports the `google.generativeai` library and then lists all available Gemini models, filtering for those that support content generation. This helps identify which models can be used for tasks like receipt analysis.

In [2]:
import google.generativeai as genai

print("Listing available Gemini models:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"Name: {m.name}, Description: {m.description}")

Listing available Gemini models:
Name: models/gemini-2.5-flash, Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
Name: models/gemini-2.5-pro, Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro
Name: models/gemini-2.0-flash, Description: Gemini 2.0 Flash
Name: models/gemini-2.0-flash-001, Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.
Name: models/gemini-2.0-flash-lite-001, Description: Stable version of Gemini 2.0 Flash-Lite
Name: models/gemini-2.0-flash-lite, Description: Gemini 2.0 Flash-Lite
Name: models/gemini-2.5-flash-preview-tts, Description: Gemini 2.5 Flash Preview TTS
Name: models/gemini-2.5-pro-preview-tts, Description: Gemini 2.5 Pro Preview TTS
Name: models/gemma-3-1b-it, Description: 
Name: models/gemma-3-4b-it, Description: 
Name: models/gemma-3-12b-it, Descripti

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


This cell initializes the `gemini_pro_vision` model using the `gemini-2.5-flash` model, preparing it for image and text content generation tasks.

In [3]:
# Initialize the Gemini Pro Vision model with the updated name
gemini_pro_vision = genai.GenerativeModel('gemini-2.5-flash')

print("Google Gemini 2.5 Flash model initialized.")

Google Gemini 2.5 Flash model initialized.


This cell loads the `naver-clova-ix/cord-v2` dataset using the `datasets` library.

In [4]:
from datasets import load_dataset

ds = load_dataset("naver-clova-ix/cord-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-b4aaeceff1d90e(…):   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00004-7dbbe248962764(…):   0%|          | 0.00/441M [00:00<?, ?B/s]

...

data/validation-00000-of-00001-cc3c5779f(…):   0%|          | 0.00/242M [00:00<?, ?B/s]

data/test-00000-of-00001-9c204eb3f4e1179(…):   0%|          | 0.00/234M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Step 0: Build df_receipt from the CORD dataset
# ------------------------------------------------------------
# Assumes you already ran something like:
# from datasets import load_dataset
# dataset = load_dataset("naver-clova-ix/cord-v2")

print("Available dataset splits:", dataset.keys())

# Use the training split by default.
# Change to "test" if you want to evaluate on that split instead.
split_name = "train"
cord_split = dataset[split_name]

print(f"Using split: {split_name}")
print(f"Number of rows in split: {len(cord_split)}")

# Convert the Hugging Face dataset split to a pandas DataFrame
df_raw = cord_split.to_pandas()

print("\nRaw columns from CORD:")
print(df_raw.columns.tolist())

NameError: name 'dataset' is not defined